### 3.2 Разбиение данных

**Входы**

- data после промежуточной очистки
- Целевая переменная loan_status

**Выходы**

- train — 70 %
- validation — 15 %
- test — 15 %

**Что делаем**

Выполняем существующее стратифицированное разбиение с random_state=42 и фиксируем независимые выборки


In [ ]:
# Только для разработки: открыть раздел можно из notebooks/ или корня проекта
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


##### Назначение обучающей, валидационной и тестовой выборок

**Утечка данных — data leakage** возникает, когда при подготовке или обучении модели используется информация, которая не должна быть доступна на этапе построения прогноза

Согласно [официальной документации scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage), тестовые данные не должны участвовать в выборе модели, отборе признаков или оценивании параметров предобработки

> As with any other type of preprocessing, feature selection should only use the training data.

Перевод: «Как и при любом другом виде предобработки, при отборе признаков следует использовать только обучающие данные»

[Официальный источник — scikit-learn, Data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

Перед статистическим отбором признаков, в том числе исключением признаков по взаимным корреляциям, необходимо выделить независимую тестовую выборку

Если решения принимаются по статистикам всего датасета, информация будущей тестовой части влияет на построение модели даже без непосредственного обучения на этих записях

**Использование тестовых данных при принятии решений может нарушить независимость итоговой оценки и сделать её излишне оптимистичной**

Следующий этап — разделение данных на три выборки

| Выборка | Назначение |
| --- | --- |
| **Train — обучающая** | Анализ зависимостей для отбора признаков, оценивание параметров предобработки и обучение модели |
| **Validation — валидационная** | Сравнение вариантов предобработки и моделей, выбор гиперпараметров |
| **Test — тестовая** | Итоговая оценка зафиксированного решения после завершения выбора модели и настроек |

Параметры преобразований определяем по train и применяем к validation и test без повторного оценивания по этим выборкам
**Замечание**

Все решения об отборе, удалении и преобразовании признаков принимаются исключительно на обучающей выборке (`train`)

Затем эти же решения применяются к `validation` и `test` без повторного анализа и пересчёта параметров на этих выборках

Таким образом, во всех трёх выборках сохраняются одинаковый набор, порядок и смысл признаков, а валидационные и тестовые данные не влияют на принятые решения


##### Параметры разделения выборки

**Scikit-learn** — библиотека Python для машинного обучения, предоставляющая инструменты предобработки данных, обучения моделей, подбора параметров и оценки качества

Функция `train_test_split` разделяет данные на непересекающиеся подвыборки

**Стратификация по целевой переменной** сохраняет приблизительно одинаковые доли классов `loan_status` в каждой выборке

Сначала выделяем train — 70 % данных, затем оставшиеся 30 % делим поровну на validation и test — по 15 % исходных данных

В обоих разбиениях используем стратификацию и `random_state=42` для воспроизводимости результата


In [ ]:
from sklearn.model_selection import train_test_split

train, remaining = train_test_split(
    data,
    test_size=0.30,
    stratify=data['loan_status'],
    random_state=42,
)

validation, test = train_test_split(
    remaining,
    test_size=0.50,
    stratify=remaining['loan_status'],
    random_state=42,
)
